# Advanced Models
In this file, a **LightGBM** model will be trained on the enriched data from the previous week and evaluated on R2, MAPE and MdAPE. Hyperparameter tuning through `RandomizedSearchCV` will be implemented to maximize performance.

### Import Libraries

In [24]:
# manipulation
import pandas as pd

# model
import lightgbm as lgb

# metrics for evaluation
from sklearn.metrics import r2_score, mean_absolute_error, median_absolute_error

# Hyperparameter tunign
from sklearn.model_selection import RandomizedSearchCV

import warnings; warnings.filterwarnings('ignore')

### Load Data
From this point going forward, the enriched data from the previous week will be used for modeling purposes. 

In [25]:
# load data
train = pd.read_csv('data/enriched_sets/tree_train_enriched.csv')
test = pd.read_csv('data/enriched_sets/tree_test_enriched.csv')

# create splits
X_train = train.drop(columns=['logClosePrice', 'ClosePrice'])
y_train = train['logClosePrice']
X_test = test.drop(columns=['logClosePrice', 'ClosePrice'])
y_test = test['logClosePrice']

In [26]:
X_train.columns

Index(['ViewYN', 'PoolPrivateYN', 'Latitude', 'Longitude', 'LivingArea',
       'CountyOrParish', 'AttachedGarageYN', 'ParkingTotal', 'YearBuilt',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'FireplaceYN',
       'Stories', 'Levels', 'NewConstructionYN', 'GarageSpaces',
       'HighSchoolDistrict', 'PostalCode', 'LotSizeSquareFeet', 'PropertyAge',
       'BedBathRatio', 'AmenityScore', 'DistrictName_TargetEnc'],
      dtype='object')

### Hyperparameter Tuning with RandomizedSearchCV

In [27]:
# initialize regressor
lgb_reg = lgb.LGBMRegressor(random_state=54)

In [28]:
# parameter values
params = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, -1],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 63, 127],
    'subsample': [0.8, 1.0]
}

# hyperparameter tuning
tuner = RandomizedSearchCV(
    estimator=lgb_reg,
    param_distributions=params,
    n_iter=15,
    cv=3,
    verbose=1,
    random_state=54,
    n_jobs=1
)

tuner.fit(X_train, y_train)
best_lgb = tuner.best_estimator_

Fitting 3 folds for each of 15 candidates, totalling 45 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001289 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2458
[LightGBM] [Info] Number of data points in the train set: 65558, number of used features: 24
[LightGBM] [Info] Start training from score 13.773003
[CV] END learning_rate=0.01, max_depth=10, n_estimators=200, num_leaves=127, subsample=1.0; total time=   4.2s
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001228 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2462
[LightGBM] [Info] Number of data points in the train set: 65559, number of used features: 24
[LightGBM] [Info] Start training from score 13.768194
[CV] END learn

In [29]:
print(f'Best LightGBM parameters: {tuner.best_params_}')

Best LightGBM parameters: {'subsample': 0.8, 'num_leaves': 127, 'n_estimators': 500, 'max_depth': -1, 'learning_rate': 0.1}


### Evaluate LightGBM

In [34]:
# predictions
preds = best_lgb.predict(X_test)

# evaluation metrics
metrics = {
    'R2': r2_score(y_test, preds),
    'MAE': mean_absolute_error(y_test, preds),
    'MdAE': median_absolute_error(y_test, preds)
}

for metric, value in metrics.items():
    print(f'{metric}: {value:.4f}')

R2: 0.9190
MAE: 0.1221
MdAE: 0.0829


### Summary
For reference, the results from `05_feature_engineering.ipynb` are as follows:
| **Model** | **R2 score** | **MAE** | **MdAE**
|---|---|---|---|
|*Linear*| 0.7812 | 0.1943 | 0.1456 |
|*Decision Tree*| 0.7801 | 0.1828 | 0.1214 |
|*Random Forest*| 0.8766 | 0.1322 | 0.0882 |

* LightGBM out-performs all 3 models above, which follows expectation
* MAE vs MdAE gap signals consistent outlier sensitivity for all models; this gap shrinks between Random Forest and LightGBM, but it still indicates that the best models struggle to predict on the few high-end luxury homes
* For reference, MAE and MdAE are in log space